In [7]:
df_customers = spark.sql("""
    SELECT * FROM bronze_customers
""")
display(df_customers)

StatementMeta(, cc9aed87-f08f-45ff-8a0c-df5fe3f16bdc, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 15d13116-085b-41a5-a392-a7236fa8a309)

In [2]:
df_customers.printSchema()

StatementMeta(, 4baed12e-0191-4fce-9102-e3f0b6d65e64, 4, Finished, Available, Finished, False)

root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- CreatedDate: timestamp (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [3]:
from pyspark.sql.functions import col 

#count total rows
print("Total rows:", df_customers.count())

# check null values in each important column
df_customers.select(
    *[
        col(column).isNull().alias(column + "_is_null")
        for column in df_customers.columns
    ]
).show()

StatementMeta(, cb07ac79-01d2-457c-928b-e28c51ddb4e1, 5, Finished, Available, Finished, False)

Total rows: 20
+------------------+--------------------+---------------+------------+---------------+-------------------+------------------------+
|CustomerID_is_null|CustomerName_is_null|Country_is_null|City_is_null|Segment_is_null|CreatedDate_is_null|LastModifiedDate_is_null|
+------------------+--------------------+---------------+------------+---------------+-------------------+------------------------+
|             false|               false|          false|       false|          false|              false|                   false|
|             false|               false|          false|       false|          false|              false|                   false|
|             false|               false|          false|       false|          false|              false|                   false|
|             false|               false|          false|       false|          false|              false|                   false|
|             false|               false|          false|    

In [8]:
#count dulplicate customerIDs
duplicate_customers = (
    df_customers.groupBy("CustomerID").count().filter("count > 1")
)
display(duplicate_customers)

StatementMeta(, cc9aed87-f08f-45ff-8a0c-df5fe3f16bdc, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cf08eff6-5b31-4edb-9c55-a6b3bbd4449a)

In [9]:
# Remove duplicate customers based on CustomerID
df_customers_silver = df_customers.dropDuplicates(["CustomerID"])

# Show the cleaned result
display(df_customers_silver)

StatementMeta(, cc9aed87-f08f-45ff-8a0c-df5fe3f16bdc, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1c2ebb1b-668d-40e7-9c94-24f72b193938)

In [4]:
print('Bronze rows:', df_customers.count())
print("Silver rows:", df_customers_silver.count())

StatementMeta(, 3952d1f7-dd5d-4778-ab67-a667b86fb5a5, 6, Finished, Available, Finished, False)

Bronze rows: 20
Silver rows: 10


In [10]:
from pyspark.sql.functions import col,trim,initcap

df_customers_silver =(
    df_customers.dropDuplicates(["CustomerID"]).filter(col("CustomerID").isNotNull())
    .withColumn("CustomerName", initcap(trim(col("CustomerName"))))
    .withColumn("Country",initcap(trim(col("Country"))))
    .withColumn("City",initcap(trim(col("City"))))
    .withColumn("Segment",initcap(trim(col("Segment"))))
)
display(df_customers_silver)

StatementMeta(, cc9aed87-f08f-45ff-8a0c-df5fe3f16bdc, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 516746c6-2771-47ba-87cb-0e2591f5570c)

In [11]:
print("Bronze rows:", df_customers.count())
print("Silver rows:", df_customers_silver.count())

df_customers_silver.printSchema()

StatementMeta(, cc9aed87-f08f-45ff-8a0c-df5fe3f16bdc, 13, Finished, Available, Finished, False)

Bronze rows: 20
Silver rows: 10
root
 |-- CustomerID: integer (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- CreatedDate: timestamp (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [12]:
# Write the cleaned customer data to the Silver Lakehouse
df_customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("LH_Sales_Silver.silver_customers")

StatementMeta(, cc9aed87-f08f-45ff-8a0c-df5fe3f16bdc, 14, Finished, Available, Finished, False)

AnalysisException: [SCHEMA_NOT_FOUND] The schema `default.Fabric-Sales-Analytics-Portfolio.LH_Sales_Bronze.LH_Sales_Silver` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a catalog, verify the current_schema() output, or qualify the name with the correct catalog.
To tolerate the error on drop use DROP SCHEMA IF EXISTS.

In [13]:
spark.sql("SHOW DATABASES").show()

StatementMeta(, cc9aed87-f08f-45ff-8a0c-df5fe3f16bdc, 15, Finished, Available, Finished, False)

+--------------------+
|           namespace|
+--------------------+
|Fabric-Sales-Anal...|
+--------------------+



In [15]:
silver_customers_path = (
    "abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc@onelake.dfs.fabric.microsoft.com/cea12ecc-9858-4f2f-8e04-108bd114e8b9/Tables/dbo/silver_customers"
)

df_customers_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_customers_path)

StatementMeta(, cc9aed87-f08f-45ff-8a0c-df5fe3f16bdc, 17, Finished, Available, Finished, False)

Bronze_products

In [2]:
df_products = spark.sql("""
    SELECT * from bronze_products
""")

display(df_products)
df_products.printSchema()

StatementMeta(, 15fb5540-2cb8-43e3-9944-10b2d5dca957, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 88ce3b3a-0985-4c5c-881d-6b9189527487)

root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [3]:
from pyspark.sql.functions import col,trim,initcap

df_products_silver = (
    df_products
    .dropDuplicates(["ProductID"])
    .filter(col("ProductID").isNotNull())
    .withColumn("ProductName",trim(col("ProductName")))
    .withColumn("Category",trim(col("Category")))
    .withColumn("SubCategory",trim(col("SubCategory")))

)
display(df_products_silver)

StatementMeta(, 15fb5540-2cb8-43e3-9944-10b2d5dca957, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3883a5f3-5d9d-440b-88f9-2ecf3dcbe988)

In [5]:
print("Bronze product count:", df_products.count())
print("Silver product count:", df_products_silver.count())
df_products_silver.printSchema()

StatementMeta(, f032def1-fb70-450a-a124-967439a70595, 7, Finished, Available, Finished, False)

Bronze product count: 12
Silver product count: 12
root
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [5]:
silver_products_path =("abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc@onelake.dfs.fabric.microsoft.com/cea12ecc-9858-4f2f-8e04-108bd114e8b9/Tables/dbo/silver_products")

df_products_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_products_path)

StatementMeta(, 15fb5540-2cb8-43e3-9944-10b2d5dca957, 7, Finished, Available, Finished, False)

Bronze_orders


In [8]:
df_orders = spark.sql("""
    SELECT * From bronze_orders
""")

display(df_orders)
df_orders.printSchema()

StatementMeta(, 15fb5540-2cb8-43e3-9944-10b2d5dca957, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5729a67a-8367-4d7c-9d00-ebed5b355a20)

root
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Status: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [9]:
from pyspark.sql.functions import col,trim,initcap

df_orders_silver=(
    df_orders
    .dropDuplicates(["OrderID"])
    .filter(col("OrderID").isNotNull())
    .filter(col("CustomerID").isNotNull())
    .withColumn("Status",initcap(trim(col("Status"))))


)

display(df_orders_silver)

StatementMeta(, 15fb5540-2cb8-43e3-9944-10b2d5dca957, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, db54865f-a172-4cbc-b371-e4a679d115a9)

In [10]:
print("Bronze order count:", df_orders.count())
print("Silver order count:", df_orders_silver.count())

df_orders_silver.printSchema()

StatementMeta(, 15fb5540-2cb8-43e3-9944-10b2d5dca957, 12, Finished, Available, Finished, False)

Bronze order count: 16
Silver order count: 16
root
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Status: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [11]:
silver_orders_path=("abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc"
    "@onelake.dfs.fabric.microsoft.com/"
    "cea12ecc-9858-4f2f-8e04-108bd114e8b9/"
    "Tables/dbo/silver_orders")

df_orders_silver  .write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_orders_path)

StatementMeta(, 15fb5540-2cb8-43e3-9944-10b2d5dca957, 13, Finished, Available, Finished, False)

OrderDetails 

In [3]:
df_orderdetails = spark.sql("""
    SELECT * FROM bronze_orderdetails
""")

display(df_orderdetails)

df_orderdetails.printSchema()

StatementMeta(, 9195b8f4-b418-408d-99dd-c1cc48b0ca31, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 27057195-0dbe-4e85-bfab-5dc24c2b9551)

root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Discount: decimal(5,2) (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [4]:
from pyspark.sql.functions import col

df_orderdetails_silver =(
    df_orderdetails
    .drop_duplicates(["OrderDetailID"])
    .filter(col("OrderDetailID").isNotNull())
    .filter(col("OrderID").isNotNull())
    .filter(col("ProductID").isNotNull())
    .filter(col("Quantity").isNotNull())
    .filter(col("Quantity")> 0)
    .filter(col("UnitPrice").isNotNull())
    .filter(col("UnitPrice") >= 0)
    .filter(col("Discount").isNotNull())
    .filter(col("Discount") >= 0)

)

display(df_orderdetails_silver)

StatementMeta(, 9195b8f4-b418-408d-99dd-c1cc48b0ca31, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 043ba391-1bfb-40cb-8206-ea29adc3bec5)

In [5]:
print("Bronze OrderDetails count:", df_orderdetails.count())
print("Silver OrderDetails count:", df_orderdetails_silver.count())

df_orderdetails_silver.printSchema()

StatementMeta(, 9195b8f4-b418-408d-99dd-c1cc48b0ca31, 7, Finished, Available, Finished, False)

Bronze OrderDetails count: 32
Silver OrderDetails count: 32
root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Discount: decimal(5,2) (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [6]:
silver_orderdetails_path = (
    "abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc"
    "@onelake.dfs.fabric.microsoft.com/"
    "cea12ecc-9858-4f2f-8e04-108bd114e8b9/"
    "Tables/dbo/silver_orderdetails")

df_orderdetails_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_orderdetails_path)

StatementMeta(, 9195b8f4-b418-408d-99dd-c1cc48b0ca31, 8, Finished, Available, Finished, False)

## Gold – Fact Sales

In [2]:
silver_base_path = (
    "abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc"
    "@onelake.dfs.fabric.microsoft.com/"
    "cea12ecc-9858-4f2f-8e04-108bd114e8b9/"
    "Tables/dbo"
)
silver_customers_path = silver_base_path + "/silver_customers"
silver_products_path = silver_base_path + "/silver_products"
silver_orders_path = silver_base_path + "/silver_orders"
silver_orderdetails_path = silver_base_path + "/silver_orderdetails"

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 4, Finished, Available, Finished, False)

In [3]:
df_customers =spark.read.format("delta").load(silver_customers_path)

df_products =spark.read.format("delta").load(silver_products_path)
df_orders = spark.read.format("delta").load(silver_orders_path)
df_orderdetails = spark.read.format("delta").load(silver_orderdetails_path)

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 5, Finished, Available, Finished, False)

In [4]:
print("Customers:", df_customers.count())
print("Products:", df_products.count())
print("Orders",df_orders.count())
print("OrderDetails:", df_orderdetails.count)

StatementMeta(, 97977053-4a7b-4fa8-8842-6a4657ae26d7, 6, Finished, Available, Finished, False)

Customers: 10
Products: 12
Orders 16
OrderDetails: 32


In [5]:
from pyspark.sql.functions import col

od = df_orderdetails.alias("od")
o = df_orders.alias("o")
p = df_products.alias("p")

df_sales = (
    od
    .join(
        o,
        col("od.OrderID") == col("o.OrderID"),
        "inner"
    )
    .join(
        p,
        col("od.ProductID") == col("p.ProductID"),
        "inner"
    )
)

display(df_sales)

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 99454f64-5ade-42f1-8b3e-5352e44e5a4f)

In [6]:
print("OrderDetails:", df_orderdetails.count())
print("Joined sales rows:", df_sales.count())

df_sales.printSchema()

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 8, Finished, Available, Finished, False)

OrderDetails: 32
Joined sales rows: 32
root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Discount: decimal(5,2) (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Status: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)



In [8]:
df_sales_clean = df_sales.select(
    col("od.OrderDetailID").alias("OrderDetailID"),
    col("od.OrderID").alias("OrderID"),
    col("od.ProductID").alias("ProductID"),
    col("o.CustomerID").alias("CustomerID"),
    col("o.OrderDate").alias("OrderDate"),
    col("od.Quantity").alias("Quantity"),
    col("od.UnitPrice").alias("UnitPrice"),
    col("od.Discount").alias("Discount"),
    col("o.Status").alias("Status"),
    col("od.LastModifiedDate").alias("LastModifiedDate"),
    col("p.ProductName").alias("ProductName"),
    col("p.Category").alias("Category"),
    col("p.SubCategory").alias("SubCategory")
)

display(df_sales_clean)

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2d9c66d1-2d07-4a87-8436-0f7f94400019)

In [9]:
print("OrderDetails:", df_orderdetails.count())
print("Joined sales rows:", df_sales_clean.count())

df_sales_clean.printSchema()

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 11, Finished, Available, Finished, False)

OrderDetails: 32
Joined sales rows: 32
root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Discount: decimal(5,2) (nullable = true)
 |-- Status: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)



In [10]:
from pyspark.sql.functions import col

df_fact_sales = (
    df_sales_clean
    .withColumn(
        "GrossSales",
        col("Quantity") * col("UnitPrice")
    )
    .withColumn(
        "NetSales",
        (col("Quantity") * col("UnitPrice")) - col("Discount")
    )
)

display(
    df_fact_sales.select(
        "OrderDetailID",
        "OrderID",
        "CustomerID",
        "ProductID",
        "Quantity",
        "UnitPrice",
        "Discount",
        "GrossSales",
        "NetSales"
    )
)

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eae30d8e-6ef3-4956-bd87-e87c274366bd)

In [11]:
df_fact_sales.printSchema()

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 13, Finished, Available, Finished, False)

root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Discount: decimal(5,2) (nullable = true)
 |-- Status: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- SubCategory: string (nullable = true)
 |-- GrossSales: decimal(21,2) (nullable = true)
 |-- NetSales: decimal(22,2) (nullable = true)



In [12]:
from pyspark.sql.functions import sum, count, min, max

df_fact_sales.select(
    count("*").alias("TotalRows"),
    min("NetSales").alias("MinimumNetSales"),
    max("NetSales").alias("MaximumNetSales"),
    sum("NetSales").alias("TotalNetSales")
).show()

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 14, Finished, Available, Finished, False)

+---------+---------------+---------------+-------------+
|TotalRows|MinimumNetSales|MaximumNetSales|TotalNetSales|
+---------+---------------+---------------+-------------+
|       32|         125.00|        2850.00|     26605.00|
+---------+---------------+---------------+-------------+



In [13]:
gold_fact_sales = df_fact_sales.select(
    "OrderDetailID",
    "OrderID",
    "CustomerID",
    "ProductID",
    "OrderDate",
    "Quantity",
    "UnitPrice",
    "Discount",
    "GrossSales",
    "NetSales",
    "Status",
    "LastModifiedDate"
)

display(gold_fact_sales)

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f478085e-d133-4adb-86c8-d3cad3bb5698)

In [14]:
gold_base_path = (
    "abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc"
    "@onelake.dfs.fabric.microsoft.com/"
    "035347ba-11fe-4f4e-a474-d142118f5ef7/"
    "Tables/dbo"
)

gold_fact_sales_path = gold_base_path + "/gold_fact_sales"

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 16, Finished, Available, Finished, False)

In [15]:
gold_fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_fact_sales_path)

StatementMeta(, 20aafdd4-77d4-45ee-aeaf-b49c970e6264, 17, Finished, Available, Finished, False)

In [1]:
silver_base_path = (
    "abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc"
    "@onelake.dfs.fabric.microsoft.com/"
    "cea12ecc-9858-4f2f-8e04-108bd114e8b9/"
    "Tables/dbo"
)

silver_customers_path = silver_base_path + "/silver_customers"
silver_products_path = silver_base_path + "/silver_products"
silver_orders_path = silver_base_path + "/silver_orders"
silver_orderdetails_path = silver_base_path + "/silver_orderdetails"

StatementMeta(, e365b2ac-d465-4e5a-ab29-13d6d7078d29, 3, Finished, Available, Finished, False)

In [5]:
df_customers = spark.read.format("delta").load(
    silver_customers_path
)

display(df_customers)

StatementMeta(, 2d1e8334-7755-49b4-984d-246f10a77432, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 84e94f4b-ae31-4165-97a4-5b1c97b337ed)

In [6]:
gold_dim_customer = df_customers.select(
    "CustomerID",
    "CustomerName",
    "Country",
    "City",
    "Segment"
)

display(gold_dim_customer)

StatementMeta(, 2d1e8334-7755-49b4-984d-246f10a77432, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, be255768-d43a-4239-981e-30f229a2c534)

In [9]:
gold_base_path = (
    "abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc"
    "@onelake.dfs.fabric.microsoft.com/"
    "035347ba-11fe-4f4e-a474-d142118f5ef7/"
    "Tables/dbo"
)
gold_dim_customer_path = gold_base_path + "/gold_dim_customer"

gold_dim_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_dim_customer_path)


StatementMeta(, 2d1e8334-7755-49b4-984d-246f10a77432, 11, Finished, Available, Finished, False)

In [2]:
df_products = spark.read.format("delta").load(
    silver_products_path
)

display(df_products)


StatementMeta(, e365b2ac-d465-4e5a-ab29-13d6d7078d29, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c1a62c8e-3c24-4c01-88ad-ce67571b78d1)

In [3]:
gold_dim_product = df_products.select(
    "ProductID",
    "ProductName",
    "Category",
    "SubCategory",
    "UnitPrice"
)

display(gold_dim_product)

StatementMeta(, e365b2ac-d465-4e5a-ab29-13d6d7078d29, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6fc3bb58-36a9-424b-8053-3b11a3ee13e0)

In [6]:
gold_base_path = (
    "abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc"
    "@onelake.dfs.fabric.microsoft.com/"
    "035347ba-11fe-4f4e-a474-d142118f5ef7/"
    "Tables/dbo"
)
gold_dim_product_path = gold_base_path + "/gold_dim_product"

gold_dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_dim_product_path)

StatementMeta(, e365b2ac-d465-4e5a-ab29-13d6d7078d29, 8, Finished, Available, Finished, False)

In [2]:
gold_base_path = (
    "abfss://863fe7fa-274e-45c0-96b7-6662f3b1adfc"
    "@onelake.dfs.fabric.microsoft.com/"
    "035347ba-11fe-4f4e-a474-d142118f5ef7/"
    "Tables/dbo"
)

gold_fact_sales_path = gold_base_path + "/gold_fact_sales"
gold_fact_sales = spark.read.format("delta").load(
    gold_fact_sales_path
)

display(gold_fact_sales)

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 52ea249c-3fa8-4844-84f2-158dbc60c23a)

In [3]:
gold_fact_sales = spark.read.format("delta").load(
    gold_fact_sales_path
)

display(gold_fact_sales)

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 14d5f24a-bc2a-4a61-9885-dbbffedc6ef4)

In [4]:
from pyspark.sql.functions import min, max

date_range = gold_fact_sales.select(
    min("OrderDate").alias("MinDate"),
    max("OrderDate").alias("MaxDate")
)

display(date_range)

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8e65a3c0-4322-42c6-8a0c-f9b1997c3f7a)

In [5]:
from pyspark.sql.functions import (
    min, max, sequence, explode,
    to_date, year, month, dayofmonth,
    dayofweek, date_format
)

date_range = gold_fact_sales.select(
    min("OrderDate").alias("MinDate"),
    max("OrderDate").alias("MaxDate")
)

date_df = date_range.select(
    explode(
        sequence(
            to_date("MinDate"),
            to_date("MaxDate")
        )
    ).alias("Date")
)

display(date_df)

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ccda9b9e-4be6-4261-9298-ca0078e8b2ee)

In [6]:
from pyspark.sql.functions import (
    year,
    month,
    dayofmonth,
    dayofweek,
    date_format,
    quarter
)

gold_dim_date = (
    date_df
    .withColumn("Year", year("Date"))
    .withColumn("MonthNumber", month("Date"))
    .withColumn("MonthName", date_format("Date", "MMMM"))
    .withColumn("Quarter", quarter("Date"))
    .withColumn("Day", dayofmonth("Date"))
    .withColumn("DayName", date_format("Date", "EEEE"))
    .withColumn("DayOfWeekNumber", dayofweek("Date"))
)

display(gold_dim_date)

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f7c12aa6-db9d-45cc-89ea-ef4abd72de53)

In [7]:
gold_dim_date_path = gold_base_path + "/gold_dim_date"

gold_dim_date.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_dim_date_path)

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 9, Finished, Available, Finished, False)

In [8]:
gold_fact_sales.printSchema()
gold_dim_date.printSchema()

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 10, Finished, Available, Finished, False)

root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Discount: decimal(5,2) (nullable = true)
 |-- GrossSales: decimal(21,2) (nullable = true)
 |-- NetSales: decimal(22,2) (nullable = true)
 |-- Status: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)

root
 |-- Date: date (nullable = false)
 |-- Year: integer (nullable = false)
 |-- MonthNumber: integer (nullable = false)
 |-- MonthName: string (nullable = false)
 |-- Quarter: integer (nullable = false)
 |-- Day: integer (nullable = false)
 |-- DayName: string (nullable = false)
 |-- DayOfWeekNumber: integer (nullable = false)



****#Add OrderDateKey to the fact table

In [9]:
from pyspark.sql.functions import to_date

gold_fact_sales = (
    gold_fact_sales
    .withColumn("OrderDateKey", to_date("OrderDate"))
)

display(
    gold_fact_sales.select(
        "OrderID",
        "OrderDate",
        "OrderDateKey",
        "NetSales"
    )
)

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d86de9b2-f036-4eae-b114-2e9a0d1392a9)

In [11]:
gold_fact_sales.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .save(gold_fact_sales_path)

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 13, Finished, Available, Finished, False)

In [12]:
gold_fact_sales.printSchema()

StatementMeta(, 7dd40326-9468-4bc8-a712-7b57fe619ae4, 14, Finished, Available, Finished, False)

root
 |-- OrderDetailID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- ProductID: integer (nullable = true)
 |-- OrderDate: timestamp (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Discount: decimal(5,2) (nullable = true)
 |-- GrossSales: decimal(21,2) (nullable = true)
 |-- NetSales: decimal(22,2) (nullable = true)
 |-- Status: string (nullable = true)
 |-- LastModifiedDate: timestamp (nullable = true)
 |-- OrderDateKey: date (nullable = true)

